<a href="https://colab.research.google.com/github/FarazTheAnalyst/Data-Scientist-Portfolio/blob/main/Resume%20Screening%20with%20RAG%20%2B%20LLM%20Project%20/rag_resume_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install faiss-cpu
!pip install PyPDF2 python-docx

In [6]:
import pandas as pd
import numpy as np
import json
import os
import pickle
import faiss
from sentence_transformers import SentenceTransformer
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import PyPDF2
import docx
import re
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

In [7]:
# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [8]:
# Load dataset
df = pd.read_csv("/content/AI_Resume_Matcher_Dataset.csv")
print(f"Loaded {len(df)} resumes")
print(f"Columns: {df.columns.tolist()}")
print(f"Sample data:\n{df.head(2)}")
print(f"\nDataset info:")
print(df.info())

Loaded 2000 resumes
Columns: ['Name', 'Experience_Years', 'Skills', 'Education', 'Applied_Job_Role']
Sample data:
          Name  Experience_Years             Skills Education Applied_Job_Role
0  Candidate_0                 1   Node.js, MongoDB       BCA   Data Scientist
1  Candidate_1                 7  JavaScript, React     B.Com  Product Manager

Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Name              2000 non-null   object
 1   Experience_Years  2000 non-null   int64 
 2   Skills            2000 non-null   object
 3   Education         2000 non-null   object
 4   Applied_Job_Role  2000 non-null   object
dtypes: int64(1), object(4)
memory usage: 78.3+ KB
None


In [9]:
# Check for missing values
print(f"\nMissing values:\n{df.isnull().sum()}")


Missing values:
Name                0
Experience_Years    0
Skills              0
Education           0
Applied_Job_Role    0
dtype: int64


TEXT EXTRACTION FUNCTIONS (CRITICAL FOR HANDLING DIFFERENT FILE TYPES)

In [10]:
# Text extraction from different formats
def extract_text_from_file(file_path):
  if file_path.endswith(".pdf"):
    with open(file_path, "rb") as file:
      reader = PyPDF2.PdfReader(file)  #read pages and text
      text = ""
      for page in reader.pages:
        text += page.extract_text()
    return text
  elif file_path.endswith(".docx"):
    doc = docx.Document(file_path)
    text = "\n".join([paragraph.text for paragraph in doc.paragraphs])
    return text
  else:
      with open(file_path, "r", encoding="utf-8", errors="ignore") as file:
        return file.read()

TEXT PREPROCESSING FUNCTIONS

In [11]:
# Preprocess and create resume text from structured data
def create_resume_text(row):
  """Create a comprehensive resume text from structured data"""
  resume_parts = []

  # Name
  if pd.notna(row.get("Name", "")):
    resume_parts.append(f"Name: {row["Name"]}")

  # Experience
  if pd.notna(row.get("Experience_Years", "")):
    resume_parts.append(f"Experience: {row['Experience_Years']} Years")

  # Skills
  if pd.notna(row.get("Skills", "")):
    skills = row["Skills"]
    if isinstance(skills, str):
      # Split skills if they're comma-separated
      skill_list = [s.strip() for s in skills.split(",")]
      resume_parts.append(f"Skills: {', '.join(skill_list)}")

  # Education
  if pd.notna(row.get("Education", "")):
    resume_parts.append(f"Education: {row["Education"]}")

  # Applied job Role
  if pd.notna(row.get("Applied_Job_Role", "")):
    resume_parts.append(f"Applied for: {row['Applied_Job_Role']}")

  return " ".join(resume_parts) if resume_parts else "No resume data available"

In [12]:
# Create processed resumes
df["Resume_Text"] = df.apply(create_resume_text, axis=1)

In [13]:
# Preprocess resume text
def preprocess_text(text):
  # remove extra whitepace
  text = re.sub(r"\s", " ", text).strip()

  # Remove special characters (keep only basic punctuation)
  text = re.sub(r"[^\w\s\.\,\-\:]", " ", text)

  return text

In [14]:
df["Processed_Text"] = df["Resume_Text"].apply(preprocess_text)

In [15]:
# create resume chunks for better retrieval
def chunk_resume(text, chunk_size=500, ovarlap=50):
  """
  Split resume text into overlaping chunks for better retrieval
  """
  if not text or len(text.strip()) == 0:
    return ["No content available"]

  words = text.split()
  chunks = []

  if len(words) == 0:
    return [text]

  for i in range(0, len(words), chunk_size - ovarlap):
    chunk = " ".join(words[i:i + chunk_size])
    if chunk.strip():
      chunks.append(chunk)
  # if no chunk were created, return the original text
  if not chunks:
    chunks = [text[:500]]

  return chunks

EMBEDDING MODEL

In [16]:
# Initialize embedding model
embedding_model = SentenceTransformer("all-MiniLm-L6-v2")
embedding_dim = embedding_model.get_sentence_embedding_dimension()
print(f"Embedding dimension: {embedding_dim}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384


VECTOR DATABASE WITH FAISS

PROCESS RESUMES FROM DATASET:

In [17]:
# Create vector database with FAISS
def create_vector_database(resume_data, embedding_model):
  all_chunks = []
  all_metadata = []

  for resume in resume_data:
    for chunk_idx, chunk in enumerate(resume["chunks"]):
      all_chunks.append(chunk)
      all_metadata.append({
          "resume_id": resume["id"],
          "chunk_idx": chunk_idx,
          "name": resume["name"],
          "experience_years": resume["experience_years"],
          "skills": resume["skills"],
          "education": resume["education"],
          "applied_job_role": resume["applied_job_role"],
          "full_text": resume["full_text"][:300] + "..." if len(resume["full_text"]) > 300 else resume["full_text"]
      })
  # Generate embeddings
  print("Generating embeddings...")
  embeddings = embedding_model.encode(all_chunks, show_progress_bar=True)
  embeddings = np.array(embeddings).astype("float32")

  # Create FAISS index
  index = faiss.IndexFlatL2(embedding_dim) #Index is created.No vectors have been added yet. L2 Eulidean distance
  index.add(embeddings)  #vectors are stored

  return index, all_chunks, all_metadata

# --- Resume data creation (moved here for robustness) ---
resume_data = []
for idx, row in tqdm(df.iterrows(), total=len(df)):
    # Extract and preprocess text
    resume_text = row['Processed_Text']

    # Create chunks
    chunks = chunk_resume(resume_text)

    resume_data.append({
        'id': idx,
        'name': row.get('Name', f'Candidate_{idx}'),
        'experience_years': row.get('Experience_Years', 0),
        'skills': row.get('Skills', ''),
        'education': row.get('Education', ''),
        'applied_job_role': row.get('Applied_Job_Role', ''),
        'full_text': resume_text,
        'chunks': chunks,
        'num_chunks': len(chunks),
        'metadata': {
            'name': row.get('Name', f'Candidate_{idx}'),
            'experience_years': row.get('Experience_Years', 0),
            'skills': row.get('Skills', ''),
            'education': row.get('Education', ''),
            'applied_job_role': row.get('Applied_Job_Role', ''),
            'file_name': f"resume_{idx}.txt"
        }
    })

print(f"\nProcessed {len(resume_data)} resumes with {sum(r['num_chunks'] for r in resume_data)} total chunks")
# --- End of resume data creation ---

# Create vector database
vector_index, chunks, metadata = create_vector_database(resume_data, embedding_model)
print(f"Created vector database with {len(chunks)} chunks")

100%|██████████| 2000/2000 [00:00<00:00, 16590.93it/s]


Processed 2000 resumes with 2000 total chunks
Generating embeddings...


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Created vector database with 2000 chunks


In [18]:
resume_data[0]

{'id': 0,
 'name': 'Candidate_0',
 'experience_years': 1,
 'skills': 'Node.js, MongoDB',
 'education': 'BCA',
 'applied_job_role': 'Data Scientist',
 'full_text': 'Name: Candidate_0 Experience: 1 Years Skills: Node.js, MongoDB Education: BCA Applied for: Data Scientist',
 'chunks': ['Name: Candidate_0 Experience: 1 Years Skills: Node.js, MongoDB Education: BCA Applied for: Data Scientist'],
 'num_chunks': 1,
 'metadata': {'name': 'Candidate_0',
  'experience_years': 1,
  'skills': 'Node.js, MongoDB',
  'education': 'BCA',
  'applied_job_role': 'Data Scientist',
  'file_name': 'resume_0.txt'}}

In [19]:
# Save vector database
faiss.write_index(vector_index, "vector_database.index")
with open("chunks.pkl", "wb") as f:
  pickle.dump((chunks, metadata), f)

print("Vecto database saved successfullyr")

Vecto database saved successfullyr


**LLM** FOR **RAG**

In [20]:
# Initialize LLM for RAG
class ResumeRAG:
    def __init__(self, model_name="microsoft/DialoGPT-medium"):
      self.tokenizer = AutoTokenizer.from_pretrained(model_name)
      self.model = AutoModelForCausalLM.from_pretrained(model_name)
      self.model.to(device)
      self.model.eval()

      self.embedding_model = embedding_model
      self.vector_index = vector_index
      self.chunks = chunks
      self.metadata = metadata

    def retrieve_relevant_chunks(self, query, top_k=5):
      # Generate query embedding
      query_embedding = self.embedding_model.encode([query])[0].astype("float32")
      # Search in FAISS
      distances, indices = self.vector_index.search(query_embedding.reshape(1, -1), top_k)
      # Get relevant chunks
      relevant_chunks = []
      for idx in indices[0]:
        if idx < len(self.chunks):
          relevant_chunks.append({
              "chunk": self.chunks[idx][:500] + "...",
              "metadata": self.metadata[idx],
              "distance": float(distances[0][list(indices[0]).index(idx)])
          })
      return relevant_chunks

    def generate_evaluation(self, job_description, resume_text):
        # Retrieve relevant chunks
        relevant_chunks = self.retrieve_relevant_chunks(job_description, top_k=5)

        # Contruct prompt
        prompt = f"""
          Job Description: {job_description[:1000]}
          Candidate Resume: {resume_text[:1500]}

          Relevant Experience from database:
          {chr(10).join([chunk["chunk"] for chunk in relevant_chunks[:3]])}

          Based on the job description and candidate resume, provide the following evaluation

          1. Overall Match Score (9, 199)
          2. key Strengths (list 3-5)
          3. key Weekneses (list 2-3)
          4. Specific Skills (list)
          5. Missing Skills(list)
          6. Overall Recommendation (yes/no)

          Evaluation:
        """

        # Generate response
        inputs = self.tokenizer(prompt, return_tensors="pt", maxt_length=1000, trunction=True).to(device)

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.inference_mode():
          outputs = self.model.generate(
              **inputs,
              max_new_tokens=500,
              temperature=0.7,
              do_sample=True,
              pad_token_id=self.tokenizer.eos_token_id
          )

        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Extract evaluation from response
        try:
            evaluation_text = response.split("Evaluation:")[1].strip()
        except:
            evaluation_text = response

        return {
            "evaluation": evaluation_text,
            "relevant_chunks": relevant_chunks
        }

# Initialize RAG system
rag_system = ResumeRAG()


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/863M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

SAVE RAG SYSTEM

In [21]:
with open("rag_system.pkl", "wb") as f:
  pickle.dump({
      "vector_index": vector_index,
      "chunks": chunks,
      "metadata": metadata,
      "embedding_model": embedding_model
  }, f)

print("RAG system saved successfully")

RAG system saved successfully


TESTING RAG SYSTEM

In [22]:
#Test the RAG system
test_job = """
We are looking for a Data Scientist with expertise in machine learning, Python, and data
analysis.
The candidate should have experience with statistical modeling and data visualization
"""
test_candidate = resume_data[0]["full_text"]
evaluation = rag_system.generate_evaluation(test_job, test_candidate)

print("\n=== Test Evaluation ===")
print(evaluation["evaluation"])
print("\n=== Relevant Chunks ===")
for chunk in evaluation["relevant_chunks"][:3]:
  print(f"- {chunk['metadata']['name']} ({chunk['metadata']['applied_job_role']}): {chunk['chunk'][:100]}...")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



=== Test Evaluation ===


=== Relevant Chunks ===
- Candidate_1641 (Data Scientist): Name: Candidate_1641 Experience: 0 Years Skills: Python, SQL, ML Education: M.Tech Applied for: Data...
- Candidate_963 (Data Scientist): Name: Candidate_963 Experience: 8 Years Skills: Python, SQL, ML Education: B.Sc in IT Applied for: D...
- Candidate_1899 (Data Scientist): Name: Candidate_1899 Experience: 9 Years Skills: Python, SQL, ML Education: B.Tech in CS Applied for...


SAMPLE VISUALIZATION

In [29]:
# Create sample visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of experience
plt.figure(figsize=(12, 6))
df["Experience_Years"].hist(bins=20, edgecolor="black")
plt.title("Experience Distribution of Candidate")
plt.xlabel("Years of Experience")
plt.ylabel("Number of Candidates")
plt.tight_layout()
plt.savefig("experiecne_deistribution.png")
plt.close()

In [30]:
# Ditribution of applied roles
if "Applied_Job_Role" in df.columns:
  plt.figure(figsize=(14, 6))
  df["Applied_Job_Role"].value_counts().head(10).plot(kind="bar")
  plt.title("Top 10 Applied Job Roles")
  plt.xlabel("Job Role")
  plt.ylabel("Number of Candidates")
  plt.xticks(rotation=45)
  plt.tight_layout()
  plt.savefig("applied_roles.png")
  plt.close()
else:
  print("No 'Applied_Job_Role' column found in the DataFrame.")

In [31]:
# Education distribution
if "Education" in df.columns:
  plt.figure(figsize=(12, 6))
  df["Education"].value_counts().plot(kind="bar")
  plt.title("Education Dsitribution")
  plt.xlabel("Education Level")
  plt.ylabel("Number of candidates")
  plt.xticks(rotation=45)
  plt.tight_layout()
  plt.savefig("education_distribution.png")
  plt.close()
else:
    print("No 'Education' column found in the DataFrame.")